# Notebook 01 — Shape Motor Performance
**Re-run any time new data arrives.** Point `DATA_DIR` at the folder with your JSON files.

### What this notebook tells you
| Audience | What they get |
|---|---|
| **User / Patient** | Which shapes you're good at and which need more practice |
| **Doctor / Clinician** | Shape-difficulty profile as a motor coordination fingerprint |
| **App maker** | Which shapes to show more often (hard ones) vs use as warm-ups (easy ones) |


In [1]:
# ── CONFIG ─────────────────────────────────────────────
DATA_DIR     = '.'          # folder with your JSON files
OUT_DIR      = 'outputs'    # figures and CSVs saved here
GROUP_LABEL  = 'Group A'    # shown in plot titles
ROLLING_WIN  = 5            # tries to smooth over in trend lines
# ───────────────────────────────────────────────────────
import os, json, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
warnings.filterwarnings('ignore')

# ── Make sure output folder exists BEFORE anything tries to write to it ──
os.makedirs(OUT_DIR, exist_ok=True)

sys.path.insert(0, os.path.dirname(os.path.abspath('hci_utils.py')))
from hci_utils import (load_board_tries, load_bd_sessions, load_piano_sessions,
                        load_piano_movements, compare_groups, save_fig,
                        SHAPE_ORDER, HAND_COLORS, GROUP_COLORS)

df = load_board_tries(DATA_DIR)
print(f"Loaded {len(df)} tries across {df['shapeType'].nunique()} shapes")
print(df['shapeType'].value_counts().to_string())


KeyError: 'startedAt'

## 1 · Per-shape summary table

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)   # safety — also done above but be explicit before every CSV

summary = (
    df.groupby('shapeType')
    .agg(
        total_tries   = ('_id',       'count'),
        success_count = ('completed', 'sum'),
        avg_accuracy  = ('accuracy',  'mean'),
        std_accuracy  = ('accuracy',  'std'),
    )
    .reset_index()
)
summary['success_rate'] = summary['success_count'] / summary['total_tries'] * 100
summary['std_accuracy']  = summary['std_accuracy'].fillna(0)

cat = pd.CategoricalDtype(categories=SHAPE_ORDER, ordered=True)
summary['shapeType'] = summary['shapeType'].astype(cat)
summary = summary.sort_values('shapeType').reset_index(drop=True)

summary.to_csv(os.path.join(OUT_DIR, 'shape_summary.csv'), index=False)
print('shape_summary.csv saved')
print(summary[['shapeType','total_tries','success_rate','avg_accuracy','std_accuracy']].to_string(index=False))


## 2 · Fig 01a — Success rate per shape

In [ ]:
labels = summary['shapeType'].tolist()
colors = ['#2a6e4f' if v >= 80 else '#ba7517' if v >= 60 else '#c84b2f'
          for v in summary['success_rate']]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(labels, summary['success_rate'], color=colors, height=0.55)
ax.axvline(80, color='#2a6e4f', lw=1, ls='--', alpha=0.5, label='80% target')
ax.axvline(60, color='#ba7517', lw=1, ls='--', alpha=0.5, label='60% threshold')
ax.set_xlabel('Success rate (%)')
ax.set_xlim(0, 115)
ax.set_title(GROUP_LABEL + ' — Shape success rate', fontweight='bold')
for bar, val in zip(bars, summary['success_rate']):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)
ax.legend(fontsize=8)
plt.tight_layout()
save_fig(fig, 'fig01a_shape_success_rate.png', OUT_DIR)
plt.show()

# ── User feedback ──────────────────────────────────────────────────────────
best  = summary.loc[summary['success_rate'].idxmax(), 'shapeType']
worst = summary.loc[summary['success_rate'].idxmin(), 'shapeType']
print('\n=== USER FEEDBACK ===')
print(f'  Best shape:  {best}  — great motor control here, use as warm-up')
print(f'  Needs work:  {worst} — focus practice sessions on this shape')

# ── Clinician signal ───────────────────────────────────────────────────────
hard = summary[summary['success_rate'] < 60]['shapeType'].tolist()
print('\n=== CLINICIAN SIGNAL ===')
if hard:
    print(f'  Shapes below 60% success: {hard}')
    print('  These require continuous wrist rotation / multi-joint coordination.')
    print('  Document these as baseline impairment markers.')
else:
    print('  All shapes above 60% — motor coordination is broadly functional.')

# ── App-maker signal ───────────────────────────────────────────────────────
print('\n=== APP-MAKER SIGNAL ===')
low_try = summary[summary['total_tries'] < 5]['shapeType'].tolist()
if low_try:
    print(f'  Under-sampled shapes (< 5 tries): {low_try}')
    print('  Increase their spawn frequency so more data is collected.')


## 3 · Fig 01b — Accuracy distribution (box plot)

In [ ]:
shapes_present = [s for s in SHAPE_ORDER if s in df['shapeType'].values]
data = [df[df['shapeType'] == s]['accuracy'].values for s in shapes_present]

fig, ax = plt.subplots(figsize=(10, 4))
bp = ax.boxplot(data, labels=shapes_present, patch_artist=True,
                medianprops=dict(color='black', lw=2))
for patch, lbl in zip(bp['boxes'], shapes_present):
    row = summary[summary['shapeType'] == lbl]
    v   = float(row['success_rate'].values[0]) if len(row) else 50
    patch.set_facecolor('#2a6e4f' if v >= 80 else '#ba7517' if v >= 60 else '#c84b2f')
    patch.set_alpha(0.6)
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 110)
ax.set_title(GROUP_LABEL + ' — Accuracy distribution per shape', fontweight='bold')
plt.tight_layout()
save_fig(fig, 'fig01b_shape_accuracy_distribution.png', OUT_DIR)
plt.show()

# High variability = inconsistent performance = clinically significant
print('\n=== CLINICIAN SIGNAL ===')
print('Shapes with high std (inconsistent performance):')
high_var = summary[summary['std_accuracy'] > 20]
if high_var.empty:
    print('  None — performance is consistent across shapes.')
else:
    for _, row in high_var.iterrows():
        print(f"  {row['shapeType']}: std={row['std_accuracy']:.1f}% "
              "(inconsistent — may indicate tremor or fatigue)")


## 4 · Fig 01c — Shape radar chart

In [ ]:
angles = np.linspace(0, 2 * np.pi, len(shapes_present), endpoint=False).tolist()
angles += angles[:1]

sr_vals = [float(summary.loc[summary['shapeType']==s,'success_rate'].values[0])/100
           if s in summary['shapeType'].values else 0 for s in shapes_present]
ac_vals = [float(summary.loc[summary['shapeType']==s,'avg_accuracy'].values[0])/100
           if s in summary['shapeType'].values else 0 for s in shapes_present]
sr_vals += sr_vals[:1]
ac_vals += ac_vals[:1]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
ax.plot(angles, sr_vals, color='#2c4fa0', lw=2, label='Success rate')
ax.fill(angles, sr_vals, color='#2c4fa0', alpha=0.15)
ax.plot(angles, ac_vals, color='#c84b2f', lw=2, label='Avg accuracy')
ax.fill(angles, ac_vals, color='#c84b2f', alpha=0.15)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(shapes_present, fontsize=9)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['25%', '50%', '75%', '100%'], fontsize=7)
ax.set_title(GROUP_LABEL + ' — Shape performance radar', fontweight='bold', pad=18)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)
plt.tight_layout()
save_fig(fig, 'fig01c_shape_radar.png', OUT_DIR)
plt.show()

print('\n=== APP-MAKER SIGNAL ===')
print('Radar shape = motor coordination fingerprint.')
print('Save one radar per session to track how the shape changes over time.')


## 5 · Fig 01d — Accuracy over time per shape

In [ ]:
df_sorted = df.sort_values('startedAt').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 5))
cmap = plt.cm.get_cmap('tab10', len(shapes_present))
for i, shape in enumerate(shapes_present):
    sub = df_sorted[df_sorted['shapeType'] == shape].copy().reset_index(drop=True)
    if len(sub) < 2:
        continue
    sub['rolling_acc'] = sub['accuracy'].rolling(ROLLING_WIN, min_periods=1).mean()
    sub['try_num']     = range(1, len(sub) + 1)
    ax.plot(sub['try_num'], sub['rolling_acc'], label=shape,
            color=cmap(i), lw=2, marker='o', markersize=3)

ax.set_xlabel(f'Try number per shape (rolling window={ROLLING_WIN})')
ax.set_ylabel('Accuracy (%)')
ax.set_title(GROUP_LABEL + ' — Accuracy over time per shape', fontweight='bold')
ax.legend(fontsize=8, ncol=2)
ax.set_ylim(0, 110)
plt.tight_layout()
save_fig(fig, 'fig01d_shape_accuracy_over_time.png', OUT_DIR)
plt.show()

print('\n=== CLINICIAN SIGNAL ===')
print('Upward slope = learning / motor recovery. Flat = plateau. Downward = fatigue.')
print('\n=== USER FEEDBACK ===')
print('Each line shows how your accuracy on that shape changes with practice.')
print('Lines trending upward = you are improving. Keep going!')


## 6 · Fig 01e — Try count vs success rate

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for _, row in summary.iterrows():
    ax.scatter(row['total_tries'], row['success_rate'], s=120, color='#2c4fa0', zorder=3)
    ax.annotate(row['shapeType'],
                (row['total_tries'], row['success_rate']),
                textcoords='offset points', xytext=(6, 3), fontsize=8)
ax.set_xlabel('Number of tries')
ax.set_ylabel('Success rate (%)')
ax.set_title(GROUP_LABEL + ' — Try count vs success rate', fontweight='bold')
ax.axhline(80, color='#2a6e4f', ls='--', lw=1, alpha=0.5)
ax.set_ylim(0, 110)
plt.tight_layout()
save_fig(fig, 'fig01e_try_count_vs_success.png', OUT_DIR)
plt.show()

print('\n=== APP-MAKER SIGNAL ===')
print('Shapes in bottom-right (many tries, low success) = stuck, consider adaptive hints.')
print('Shapes in top-left (few tries, high success) = too easy, increase challenge.')
